# Клиенты торгового эквайринга

Один файл Excel, три листа. ИНН из 9 цифр дополняется до 10, из 11 — до 12. Если после нормализации филиала осталось только `РФ`, пишется `ЦРМБ`.

`С ТЭ на 01.09.2026` — одна строка на договор SA, открытый в этот день, с привязкой `P`. Колонки: ИНН, ЦФТ ID, номер договора, дата открытия, дата закрытия, РФ. Договор без ЦФТ ID на листе остаётся.

`Без ТЭ на 01.09.2026` — одна строка на ЦФТ ID. Организации `CL_ORG` с ИНН, которых нет среди ЦФТ ID первого листа. Кроме ИНН и ЦФТ ID только РФ с карточки клиента.

`Подключили ТЭ 06-08.2026` — одна строка на договор SA, открытый с 01.06.2026 по 31.08.2026, те же шесть колонок. Привязка `P` открыта в день открытия. Договор, закрытый до 01.09.2026, остаётся.

Смените даты и запустите ячейки по порядку. Файл ляжет в `/home/jovyan/documents/Equaring`.


In [ ]:
import getpass
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

AS_OF_DATE = '2026-09-01'
PERIOD_START = '2026-06-01'
PERIOD_END = '2026-08-31'
OUT_DIR = Path('/home/jovyan/documents/Equaring')

as_of = pd.to_datetime(AS_OF_DATE).strftime('%Y-%m-%d')
period_start = pd.to_datetime(PERIOD_START).strftime('%Y-%m-%d')
period_end = pd.to_datetime(PERIOD_END).strftime('%Y-%m-%d')

def ru_date(iso):
    year, month, day = iso.split('-')
    return '%s.%s.%s' % (day, month, year)

sheet_with = 'С ТЭ на %s' % ru_date(as_of)
sheet_without = 'Без ТЭ на %s' % ru_date(as_of)
sheet_opened = 'Подключили ТЭ %s-%s.%s' % (period_start[5:7], period_end[5:7], period_start[:4])
print(as_of, period_start, period_end)

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
import re

def normalize_filial_rf(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\s+', ' ', str(v)).strip(' ,;.-')
    if not s or s.lower() in ('none', 'nan', '<na>'):
        return None
    m = re.search(r'(?i)рф', s)
    if m:
        s = s[:m.end()]
    s = re.sub(r'(?i)\b(ао|пао|оао|зао)\b.*$', '', s).strip(' ,;.-')
    if not s:
        return None
    s = s[:1].upper() + s[1:].lower()
    s = re.sub(r'(?i)рф', 'РФ', s)
    if s == 'РФ':
        return 'ЦРМБ'
    return s

def _text(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.lower() in ('', 'none', 'nan', '<na>'):
        return None
    return s

def to_contracts(raw):
    out = pd.DataFrame({
        'ИНН': raw['inn'].map(_text),
        'ЦФТ ID': raw['cft_id'].map(_text),
        'Номер договора': raw['c_agr_number'].map(_text),
        'Дата открытия договора': raw['d_valid_from'],
        'Дата закрытия договора': raw['d_valid_to'],
        'РФ, в котором обслуживается клиент': raw['filial_rf'].apply(normalize_filial_rf),
    })
    return out.sort_values(['Номер договора', 'ИНН']).reset_index(drop=True)

def contracts_sql(sa_body):
    return f'''
WITH sa AS (
{sa_body}
),
ocrm AS (
  SELECT inn, row_id
  FROM (
    SELECT
      s.inn,
      soe.row_id,
      row_number() OVER (
        PARTITION BY s.inn
        ORDER BY soe.created DESC, soe.row_id DESC
      ) AS rn
    FROM (SELECT DISTINCT inn FROM sa) s
    JOIN ocrm_ul.s_org_ext soe
      ON CASE length(regexp_replace(soe.x_inn, '[^0-9]', ''))
           WHEN 9 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 10, '0')
           WHEN 11 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 12, '0')
           ELSE regexp_replace(soe.x_inn, '[^0-9]', '')
         END = s.inn
    WHERE soe.x_removed_flg = 'N'
      AND soe.x_duplicate_flg = 'N'
  ) t
  WHERE rn = 1
),
cdi AS (
  SELECT inn, cdi_id
  FROM (
    SELECT
      o.inn,
      e.party_id AS cdi_id,
      row_number() OVER (PARTITION BY o.inn ORDER BY e.party_id DESC) AS rn
    FROM ocrm o
    JOIN cdiul.ext_id_org e
      ON e.cmo_ext_party_source_id = o.row_id
     AND e.cmo_ext_source_system LIKE 'OCRM%'
  ) t
  WHERE rn = 1
),
cft AS (
  SELECT c.inn, e.cmo_ext_party_source_id AS cft_id
  FROM cdi c
  JOIN cdiul.ext_id_org e
    ON e.party_id = c.cdi_id
   AND e.cmo_ext_source_system LIKE 'CFT%'
),
filial AS (
  SELECT cft_id, filial_rf
  FROM (
    SELECT
      c.cft_id,
      br.c_shortlabel AS filial_rf,
      row_number() OVER (
        PARTITION BY c.cft_id
        ORDER BY CASE WHEN m.c_depart IS NULL THEN 1 ELSE 0 END, m.id DESC
      ) AS rn
    FROM (SELECT DISTINCT cft_id FROM cft) c
    JOIN ods.scd1_z_r2_ip_merchants m
      ON cast(m.c_cl_org AS string) = c.cft_id
    LEFT JOIN ods.scd1_z_depart dep ON dep.id = m.c_depart
    LEFT JOIN ods.scd1_z_branch br ON br.id = dep.c_filial
    WHERE coalesce(m.ods_deleted_flg, '0') <> '1'
  ) t
  WHERE rn = 1
),
picked AS (
  SELECT
    s.inn,
    c.cft_id,
    s.c_agr_number,
    s.d_valid_from,
    s.d_valid_to,
    f.filial_rf,
    row_number() OVER (
      PARTITION BY s.n_agr
      ORDER BY CASE WHEN f.filial_rf IS NULL THEN 1 ELSE 0 END, c.cft_id
    ) AS rn
  FROM sa s
  LEFT JOIN cft c ON c.inn = s.inn
  LEFT JOIN filial f ON f.cft_id = c.cft_id
)
SELECT inn, cft_id, c_agr_number, d_valid_from, d_valid_to, filial_rf
FROM picked
WHERE rn = 1
'''

sql_asof = contracts_sql(f'''
  SELECT DISTINCT
    a.n_agr,
    a.c_agr_number,
    a.d_valid_from,
    a.d_valid_to,
    CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c.c_inn, '[^0-9]', '')
    END AS inn
  FROM ods_alpha.scd1_agreements a
  JOIN ods_alpha.scd1_companies c
    ON c.n_cmp = a.n_cmp_client
  WHERE upper(trim(a.acq_class)) = 'SA'
    AND a.d_valid_from <= '{as_of}'
    AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{as_of}')
    AND coalesce(a.ods_deleted_flg, '0') <> '1'
    AND coalesce(c.ods_deleted_flg, '0') <> '1'
    AND c.c_inn IS NOT NULL
    AND EXISTS (
      SELECT 1
      FROM ods_alpha.scd1_agr_terms t
      WHERE t.n_agr = a.n_agr
        AND t.d_valid_from <= '{as_of}'
        AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{as_of}')
        AND upper(trim(t.cf_ter_type)) = 'P'
        AND coalesce(t.ods_deleted_flg, '0') <> '1'
    )
''')

with imp:
    imp.execute('set MEM_LIMIT=16g')
    with_te = to_contracts(imp.fetch(sql_asof))
print(sheet_with, len(with_te))
display(with_te.head(20))


In [ ]:
sql_orgs = '''
SELECT cft_id, inn, filial_rf
FROM (
  SELECT
    cast(cl.id AS string) AS cft_id,
    CASE length(regexp_replace(cl.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(cl.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(cl.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(cl.c_inn, '[^0-9]', '')
    END AS inn,
    br.c_shortlabel AS filial_rf,
    row_number() OVER (
      PARTITION BY cl.id
      ORDER BY CASE WHEN br.c_shortlabel IS NULL THEN 1 ELSE 0 END
    ) AS rn
  FROM ods.scd1_z_client cl
  LEFT JOIN ods.scd1_z_branch br
    ON br.id = cl.c_filial
  WHERE cl.class_id = 'CL_ORG'
    AND cl.c_inn IS NOT NULL
) t
WHERE rn = 1
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    orgs = imp.fetch(sql_orgs)

all_orgs = pd.DataFrame({
    'ИНН': orgs['inn'].map(_text),
    'ЦФТ ID': orgs['cft_id'].map(_text),
    'РФ, в котором обслуживается клиент': orgs['filial_rf'].apply(normalize_filial_rf),
})
all_orgs = all_orgs[all_orgs['ЦФТ ID'].notna()].drop_duplicates('ЦФТ ID')
te_ids = set(with_te['ЦФТ ID'].dropna())
without_te = (
    all_orgs.loc[~all_orgs['ЦФТ ID'].isin(te_ids)]
    .sort_values('ЦФТ ID')
    .reset_index(drop=True)
)
print(sheet_without, len(without_te))
display(without_te.head(20))


In [ ]:
sql_opened = contracts_sql(f'''
  SELECT n_agr, c_agr_number, d_valid_from, d_valid_to, inn
  FROM (
    SELECT
      a.n_agr,
      a.c_agr_number,
      a.d_valid_from,
      a.d_valid_to,
      CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
        WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
        WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
        ELSE regexp_replace(c.c_inn, '[^0-9]', '')
      END AS inn
    FROM ods_alpha.scd1_agreements a
    JOIN ods_alpha.scd1_companies c
      ON c.n_cmp = a.n_cmp_client
    WHERE upper(trim(a.acq_class)) = 'SA'
      AND a.d_valid_from >= '{period_start}'
      AND a.d_valid_from <= '{period_end}'
      AND coalesce(a.ods_deleted_flg, '0') <> '1'
      AND coalesce(c.ods_deleted_flg, '0') <> '1'
      AND c.c_inn IS NOT NULL
      AND EXISTS (
        SELECT 1
        FROM ods_alpha.scd1_agr_terms t
        WHERE t.n_agr = a.n_agr
          AND t.d_valid_from <= a.d_valid_from
          AND t.d_valid_to IS NULL
          AND upper(trim(t.cf_ter_type)) = 'P'
          AND coalesce(t.ods_deleted_flg, '0') <> '1'
      )
    UNION ALL
    SELECT
      a.n_agr,
      a.c_agr_number,
      a.d_valid_from,
      a.d_valid_to,
      CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
        WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
        WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
        ELSE regexp_replace(c.c_inn, '[^0-9]', '')
      END AS inn
    FROM ods_alpha.scd1_agreements a
    JOIN ods_alpha.scd1_companies c
      ON c.n_cmp = a.n_cmp_client
    WHERE upper(trim(a.acq_class)) = 'SA'
      AND a.d_valid_from >= '{period_start}'
      AND a.d_valid_from <= '{period_end}'
      AND coalesce(a.ods_deleted_flg, '0') <> '1'
      AND coalesce(c.ods_deleted_flg, '0') <> '1'
      AND c.c_inn IS NOT NULL
      AND EXISTS (
        SELECT 1
        FROM ods_alpha.scd1_agr_terms t
        WHERE t.n_agr = a.n_agr
          AND t.d_valid_from <= a.d_valid_from
          AND t.d_valid_to >= a.d_valid_from
          AND upper(trim(t.cf_ter_type)) = 'P'
          AND coalesce(t.ods_deleted_flg, '0') <> '1'
      )
  ) u
''')

with imp:
    imp.execute('set MEM_LIMIT=16g')
    opened = to_contracts(imp.fetch(sql_opened))
print(sheet_opened, len(opened))
display(opened.head(20))


In [ ]:
out = OUT_DIR / ('trade_acquiring_clients_%s.xlsx' % as_of)
out.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    with_te.to_excel(writer, index=False, sheet_name=sheet_with)
    without_te.to_excel(writer, index=False, sheet_name=sheet_without)
    opened.to_excel(writer, index=False, sheet_name=sheet_opened)
print(out)
